# Synthetic Data Generator

## Project Overview

Generating high-quality datasets can be expensive and time-consuming, especially when real-world data is limited, sensitive, difficult to collect, or expensive to label.

This project explores how **Generative AI** can be used to create useful, structured synthetic datasets from natural-language descriptions.

The goal is to build a simple tool where a user describes the type of data they need, specifies the number of examples, and receives a validated dataset that can be previewed or downloaded for experimentation, prototyping, or model development.

This project also serves as a hands-on exploration of key Generative AI concepts, including:

* Hugging Face pipelines
* Tokenization and Transformers
* Model inference through APIs
* Prompt design
* Structured output generation
* Synthetic data validation
* Gradio interfaces

## Core Idea

The user should not need to understand models, tokenizers, prompts, or generation parameters.

Instead, the system hides that complexity behind a simple workflow:

```text
┌──────────────────────────────┐
│   User describes dataset     │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│    Generation Pipeline       │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│    Generative AI Model       │
│   (Local Model / API)        │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│  Parse & Structure Output    │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│     Validate & Clean Data    │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│      Generated Dataset       │
└──────────────┬───────────────┘
               │
          ┌────┴────┐
          ▼         ▼
     ┌────────┐  ┌──────────┐
     │Preview │  │ Download │
     └────────┘  └──────────┘
```

## System Overview

The system accepts a natural-language description of the dataset and converts it into structured synthetic data.

```text
                  ┌─────────────────┐
                  │      User       │
                  │ Dataset Request │
                  └────────┬────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │ Prompt Builder  │
                  └────────┬────────┘
                           │
                           ▼
              ┌──────────────────────────┐
              │     Model Inference      │
              │                          │
              │  Hugging Face Pipeline   │
              │  Transformer + Tokenizer │
              │  API-based Model         │
              └────────────┬─────────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │ Output Parsing  │
                  └────────┬────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │ Data Validation │
                  └────────┬────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │ Synthetic Data  │
                  └────────┬────────┘
                           │
                    ┌──────┴──────┐
                    ▼             ▼
               ┌─────────┐   ┌──────────┐
               │ Preview │   │ Download │
               └─────────┘   └──────────┘
```

## Example

A user could enter:

> Create 100 realistic customer support records for a telecom company. Each record should contain the customer's message, issue type, sentiment, and resolution.

The system should transform that description into a structured dataset such as:

| Customer Message                           | Issue Type     | Sentiment  | Resolution                        |
| ------------------------------------------ | -------------- | ---------- | --------------------------------- |
| My internet has been down since yesterday. | Network outage | Frustrated | Technical support dispatched      |
| I was charged twice for my bundle.         | Billing        | Concerned  | Duplicate charge refunded         |
| How can I change my mobile plan?           | Plan change    | Neutral    | Plan change instructions provided |

## Project Objectives

1. Generate synthetic data from natural-language descriptions.
2. Experiment with different models and inference approaches.
3. Explore how tokenization and Transformer models work underneath Hugging Face pipelines.
4. Generate structured outputs suitable for dataset creation.
5. Validate and clean generated records.
6. Build a simple **Gradio interface** for non-technical users.
7. Evaluate the usefulness and quality of the generated data.

## Final Goal

Build a simple and practical **Synthetic Data Generator** that hides the complexity of Generative AI from the user while providing hands-on experience with the underlying technologies and techniques.

In [2]:
# !pip install bitsandbytes accelerate transformers==4.57.6

In [3]:
# libraries
import os
import json
import pandas as pd
import torch

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import InferenceClient, login
import gradio as gr

from dotenv import load_dotenv

In [ ]:
HF_TOKEN='HF TOKEN'

load_dotenv(override=True)
# api_key = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    print("API Keys not found")
else:
    print("API Key found and in use")

login(token=HF_TOKEN)
print('Login Successful')

API Key found and in use
Login Successful


In [5]:
LLAMA = 'meta-llama/Llama-3.1-8B-Instruct'
PHI = 'microsoft/Phi-4-mini-instruct'
GEMMA = 'google/gemma-3-270m-it'
QWEN = 'Qwen3-4B-Instruct-2507'
DEEPSEEK = 'deepseek-ai/DeepSeek-RI-Distill-Qwen-1.5B'


# models can be here as we experiment with them
MODELS = {
    'llama': LLAMA,
    'phi': PHI,
    'gemma': GEMMA,
    'qwen': QWEN,
    'deepseek': DEEPSEEK
}

MODELS

{'llama': 'meta-llama/Llama-3.1-8B-Instruct',
 'phi': 'microsoft/Phi-4-mini-instruct',
 'gemma': 'google/gemma-3-270m-it',
 'qwen': 'Qwen3-4B-Instruct-2507',
 'deepseek': 'deepseek-ai/DeepSeek-RI-Distill-Qwen-1.5B'}

## Prompt Design

The user will describe the dataset they want in natural language. Our system will then transform that request into a prompt that instructs the language model to generate structured synthetic data.

The prompt needs to communicate:

* What type of dataset should be generated.
* How many records are required.
* What fields each record should contain.
* The expected output format.
* Requirements for realistic and diverse examples.
* Instructions to avoid unnecessary explanations or additional text.

The dataset itself is **not hardcoded into the application**. The same generation pipeline should work with different types of data.

For example, the user could request:

> Generate 50 product reviews with the product category, review text, rating, and sentiment.

Or:

> Generate 100 patient appointment records containing age, department, appointment type, and appointment outcome.

The prompt builder should be able to handle both requests using the same underlying logic.


In [6]:
# function to generate the prompt
def build_prompt(dataset_request, num_records):
    prompt = f"""
        You are a synthetic data generation system.

        Generate {num_records} synthetic data records based on the following request:

        {dataset_request}

        Output requirements:
        - Return ONLY valid JSON.
        - Return a JSON array of objects.
        - Do NOT write Python code.
        - Do NOT include explanations, comments, or markdown.
        - Every object must contain exactly the fields requested.
        - Make the records realistic and diverse.
        - Avoid repetitive examples.
        - Ensure the fields are logically consistent with one another.
    """

    return prompt

In [7]:
dataset_request = """
Generate customer support records for a telecommunications company.

Each record should contain:
- customer_message
- issue_type
- sentiment
- resolution
"""

prompt = build_prompt(dataset_request, 10)

print(prompt)


        You are a synthetic data generation system.

        Generate 10 synthetic data records based on the following request:

        
Generate customer support records for a telecommunications company.

Each record should contain:
- customer_message
- issue_type
- sentiment
- resolution


        Output requirements:
        - Return ONLY valid JSON.
        - Return a JSON array of objects.
        - Do NOT write Python code.
        - Do NOT include explanations, comments, or markdown.
        - Every object must contain exactly the fields requested.
        - Make the records realistic and diverse.
        - Avoid repetitive examples.
        - Ensure the fields are logically consistent with one another.
    


## First Generation Experiment — Hugging Face Pipeline

The first version will use the Hugging Face `text-generation` pipeline.

At this stage, we are intentionally keeping the implementation simple:

1. Create a dataset request.
2. Convert the request into a generation prompt.
3. Pass the prompt to the Llama model through a Hugging Face pipeline.
4. Inspect the generated output.

We will improve the output format and reliability in later sections.


In [9]:
# import gc
# import torch

# gc.collect()

# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     torch.cuda.ipc_collect()

In [8]:
# creating the pipeline
generator = pipeline(
    'text-generation',
    model=LLAMA,
    device='cuda'
)


result = generator(
    prompt,
    max_new_tokens=5000,
    do_sample=True,
    temperature=0.8,
    return_full_text=False
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [9]:
generated_text = result[0]['generated_text']

generated_text

' [{"customer_message": "I am having trouble with my internet connection.", "issue_type": "connection", "sentiment": "negative", "resolution": "reset router"}, {"customer_message": "I have a question about my bill.", "issue_type": "billing", "sentiment": "neutral", "resolution": "provide billing information"}, {"customer_message": "My phone is not working.", "issue_type": "device", "sentiment": "negative", "resolution": "replace phone"}, {"customer_message": "I am experiencing issues with my voicemail.", "issue_type": "service", "sentiment": "negative", "resolution": "reset voicemail"}, {"customer_message": "I would like to upgrade my plan.", "issue_type": "plan", "sentiment": "positive", "resolution": "upgrade plan"}, {"customer_message": "I need help setting up my new device.", "issue_type": "device", "sentiment": "positive", "resolution": "provide setup instructions"}, {"customer_message": "I am having trouble hearing the other person on my calls.", "issue_type": "quality", "sentime

## Parse and Validate the Generated Data

The model is now producing structured JSON, which allows us to process the generated output programmatically.

However, LLM-generated data cannot be assumed to be perfectly consistent. The output may contain:

- Unexpected fields
- Missing fields
- Duplicate records
- Invalid JSON
- Inconsistent values

We therefore need a validation step before treating the generated output as a dataset.

The validation process will:

1. Parse the model output as JSON.
2. Check that the result is a list of records.
3. Verify that every record contains the required fields.
4. Remove unexpected fields.
5. Detect duplicate records.
6. Convert the validated records into a Pandas DataFrame.

In [10]:
print("Length:", len(generated_text))
print("Number of '[':", generated_text.count("["))
print("Number of ']':", generated_text.count("]"))

Length: 21143
Number of '[': 15
Number of ']': 16


In [16]:
# for i, char in enumerate(generated_text):
#     if char == "[":
#         print("Opening [ at:", i)

In [17]:
# for i, char in enumerate(generated_text):
#     if char == "]":
#         print("Closing ] at:", i)

In [18]:
# print(generated_text[:100])
# print("-----")
# print(generated_text[-100:])

In [11]:
EXPECTED_FIELDS = [
    "customer_message",
    "issue_type",
    "sentiment",
    "resolution"
]

def parse_generated_json(text):
    text = text.strip()

    # Find the beginning of the JSON array
    start = text.find("[")

    if start == -1:
        raise ValueError("No JSON array found in model output")

    # Parse the first complete JSON value
    decoder = json.JSONDecoder()
    data, _ = decoder.raw_decode(text[start:])

    if not isinstance(data, list):
        raise ValueError("Expected a list of records")

    return data

In [12]:
generated_data = parse_generated_json(generated_text)

# print(type(generated_data))
# print(len(generated_data))

In [13]:
for i, record in enumerate(generated_data):
    missing = set(EXPECTED_FIELDS) - set(record.keys())
    unexpected = set(record.keys()) - set(EXPECTED_FIELDS)

    if missing:
        print(f"Record {i+1} - Missing fields: {missing}")

    if unexpected:
        print(f"Record {i+1} - Unexpected fields: {unexpected}")

In [14]:
validated_data = [
    {field: record.get(field) for field in EXPECTED_FIELDS}
    for record in generated_data
]

dataframe = pd.DataFrame(validated_data)

dataframe

,customer_message,issue_type,sentiment,resolution
0,I am having trouble with my internet connection.,connection,negative,reset router
1,I have a question about my bill.,billing,neutral,provide billing information
2,My phone is not working.,device,negative,replace phone
3,I am experiencing issues with my voicemail.,service,negative,reset voicemail
4,I would like to upgrade my plan.,plan,positive,upgrade plan
5,I need help setting up my new device.,device,positive,provide setup instructions
6,I am having trouble hearing the other person o...,quality,negative,check audio settings
7,I want to cancel my service.,service,negative,cancel service
8,I am having trouble accessing my account online.,account,negative,reset password
9,I would like to add a new line to my account.,account,positive,add new line


## Dataset Validation & Evaluation

LLM-generated data should not be assumed to be correct simply because the model followed the requested format.

Before using the generated records, we perform a set of automated validation checks.

The validation checks focus on four main areas:

1. **Record count**  
   Verify that the number of generated records matches the requested number.

2. **Missing values**  
   Check for missing or empty values in required fields.

3. **Duplicates**  
   Identify exact duplicate records that may reduce the diversity of the dataset.

4. **Basic consistency**  
   Verify that:
   - all required fields are present,
   - text fields are not empty,
   - categorical values use expected labels,
   - each record has the expected data types.

These checks provide a basic quality gate before the generated data is converted into a final dataset.

In [16]:
VALID_SENTIMENTS = {
    "Positive",
    "Negative",
    "Neutral"
}

In [18]:
def evaluate_dataset(data, expected_records):
    df = pd.DataFrame(data)

    report = {}

    # -----------------------------
    # 1. Record count
    # -----------------------------
    report["expected_records"] = expected_records
    report["actual_records"] = len(df)
    report["record_count_ok"] = len(df) == expected_records

    # -----------------------------
    # 2. Check that data exists
    # -----------------------------
    report["has_data"] = not df.empty

    if df.empty:
        report["missing_values"] = {}
        report["empty_values"] = {}
        report["duplicate_records"] = 0
        report["has_duplicates"] = False
        report["columns"] = []
        return report

    # -----------------------------
    # 3. Columns
    # -----------------------------
    report["columns"] = list(df.columns)

    # -----------------------------
    # 4. Missing values
    # -----------------------------
    missing_values = df.isna().sum()

    report["missing_values"] = missing_values.to_dict()
    report["has_missing_values"] = missing_values.sum() > 0

    # -----------------------------
    # 5. Empty strings
    # -----------------------------
    empty_values = (
        df.map(
            lambda x: isinstance(x, str) and not x.strip()
        )
        .sum()
    )

    report["empty_values"] = empty_values.to_dict()
    report["has_empty_values"] = empty_values.sum() > 0

    # -----------------------------
    # 6. Duplicates
    # -----------------------------
    duplicate_count = df.duplicated().sum()

    report["duplicate_records"] = int(duplicate_count)
    report["has_duplicates"] = duplicate_count > 0

    # -----------------------------
    # 7. Schema consistency
    # -----------------------------
    expected_fields = set(data[0].keys())
    inconsistent_records = []

    for i, record in enumerate(data):
        record_fields = set(record.keys())

        if record_fields != expected_fields:
            inconsistent_records.append({
                "record": i + 1,
                "missing_fields": list(
                    expected_fields - record_fields
                ),
                "extra_fields": list(
                    record_fields - expected_fields
                )
            })

    report["schema_consistent"] = len(inconsistent_records) == 0
    report["inconsistent_records"] = inconsistent_records

    return report

In [19]:
evaluation = evaluate_dataset(
    generated_data,
    expected_records=10
)

evaluation

{'expected_records': 10,
 'actual_records': 10,
 'record_count_ok': True,
 'has_data': True,
 'columns': ['customer_message', 'issue_type', 'sentiment', 'resolution'],
 'missing_values': {'customer_message': 0,
  'issue_type': 0,
  'sentiment': 0,
  'resolution': 0},
 'has_missing_values': np.False_,
 'empty_values': {'customer_message': 0,
  'issue_type': 0,
  'sentiment': 0,
  'resolution': 0},
 'has_empty_values': np.False_,
 'duplicate_records': 0,
 'has_duplicates': np.False_,
 'schema_consistent': True,
 'inconsistent_records': []}

In [20]:
def format_evaluation_report(report):
    return (
        f"**Dataset Evaluation**\n\n"
        f"- **Records:** {report['actual_records']} / "
        f"{report['expected_records']} "
        f"{'✅' if report['record_count_ok'] else '❌'}\n"
        f"- **Columns:** {len(report['columns'])}\n"
        f"- **Missing values:** "
        f"{'Yes ❌' if report['has_missing_values'] else 'None ✅'}\n"
        f"- **Empty values:** "
        f"{'Yes ❌' if report['has_empty_values'] else 'None ✅'}\n"
        f"- **Duplicate records:** {report['duplicate_records']}\n"
        f"- **Schema consistency:** "
        f"{'Valid ✅' if report['schema_consistent'] else 'Issues found ❌'}"
    )

In [22]:
format_evaluation_report(evaluation)

'**Dataset Evaluation**\n\n- **Records:** 10 / 10 ✅\n- **Columns:** 4\n- **Missing values:** None ✅\n- **Empty values:** None ✅\n- **Duplicate records:** 0\n- **Schema consistency:** Valid ✅'

## Dynamic Model Pipeline

The application supports multiple language models through the `MODELS` dictionary.

Rather than creating a separate pipeline for each model, the application dynamically creates a pipeline based on the model selected by the user.

Pipelines are cached so that a model only needs to be loaded once during the application session.

In [23]:
from functools import lru_cache


@lru_cache(maxsize=None)
def get_generator(model_name):
    return pipeline(
        "text-generation",
        model=model_name,
        device="cuda"
    )

In [46]:
# generator = get_generator(MODELS["phi"])

# result = generator(
#     prompt,
#     max_new_tokens=500,
#     do_sample=True,
#     temperature=0.8,
#     return_full_text=False
# )


# result[0]["generated_text"]

In [24]:
def generate_dataset(dataset_request, num_records, model_key):

    model_name = MODELS[model_key]

    generator = get_generator(model_name)

    prompt = build_prompt(
        dataset_request,
        num_records
    )

    result = generator(
        prompt,
        max_new_tokens=5000,
        do_sample=True,
        temperature=0.8,
        return_full_text=False
    )

    generated_text = result[0]["generated_text"]

    generated_data = parse_generated_json(generated_text)

    evaluation = evaluate_dataset(
        generated_data,
        expected_records=num_records
    )

    df = pd.DataFrame(generated_data)

    return df, evaluation

In [25]:
df, evaluation = generate_dataset(
    dataset_request="""
    Generate customer support records for a telecommunications company.

    Each record should contain:
    - customer_message
    - issue_type
    - sentiment
    - resolution
    """,
    num_records=10,
    model_key="llama"
)

df

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


,customer_message,issue_type,sentiment,resolution
0,I'm experiencing issues with my internet conne...,Technical,Negative,Restarted the router.
1,My phone bill is incorrect.,Billing,Neutral,Reviewed and corrected the bill.
2,My service was cut off without notice.,Service,Negative,Reconnected the service.
3,I'm having trouble with my voicemail.,Technical,Negative,Reset the voicemail settings.
4,I want to change my phone plan.,Service,Positive,Upgraded the plan.
5,I'm getting wrong numbers on my bill.,Billing,Negative,Verified and corrected the numbers.
6,My internet speed is slow.,Technical,Negative,Optimized the internet settings.
7,I'm missing a feature on my phone.,Service,Positive,Added the feature.
8,I'm unhappy with my customer service experience.,Service,Negative,Provided additional training to the customer s...
9,I need help setting up my new device.,Technical,Positive,Provided step-by-step instructions.


In [54]:
# print_evaluation_report(evaluation)

## Gradio Interface

The final interface hides the underlying complexity of model loading, prompt construction, generation, parsing, and validation.

The user only needs to describe the dataset they want, specify the number of records, select a model, and generate the dataset.

In [26]:
import gradio as gr


def generate_for_ui(dataset_request, num_records, model_key):

    try:
        df, evaluation = generate_dataset(
            dataset_request,
            int(num_records),
            model_key
        )

        report = format_evaluation_report(evaluation)

        return df, report

    except Exception as e:
        return pd.DataFrame(), f"### Generation failed\n\n`{str(e)}`"

In [27]:
with gr.Blocks() as app:

    gr.Markdown(
        "# Synthetic Data Generator\n"
        "Describe the dataset you need and generate synthetic data."
    )

    dataset_request = gr.Textbox(
        label="What data do you need?",
        placeholder=(
            "Example: Generate customer support records for a "
            "telecommunications company with customer message, "
            "issue type, sentiment, and resolution."
        ),
        lines=5
    )

    with gr.Row():

        num_records = gr.Number(
            label="Number of records",
            value=10,
            precision=0
        )

        model_key = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="llama",
            label="Model"
        )

    generate_button = gr.Button(
        "Generate Dataset",
        variant="primary"
    )

    output_dataframe = gr.Dataframe(
        label="Generated Dataset",
        interactive=False
    )

    evaluation_output = gr.Markdown(
        label="Dataset Evaluation"
    )

    generate_button.click(
        fn=generate_for_ui,
        inputs=[
            dataset_request,
            num_records,
            model_key
        ],
        outputs=[
            output_dataframe,
            evaluation_output
        ]
    )

In [28]:
app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d98e6c1b2bc14c3b28.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


                    ┌─────────────────────┐
                    │        User         │
                    └──────────┬──────────┘
                               │
                               ▼
                    Dataset Description
                               │
                               ▼
                     Model Selection
                               │
                               ▼
                    ┌─────────────────────┐
                    │   MODELS dictionary │
                    └──────────┬──────────┘
                               │
                               ▼
                     Dynamic Pipeline
                               │
                               ▼
                        Prompt Builder
                               │
                               ▼
                       LLM Generation
                               │
                               ▼
                         JSON Parsing
                               │
                               ▼
                       Data Validation
                               │
                               ▼
                     ┌─────────────────┐
                     │ Pandas DataFrame│
                     └────────┬────────┘
                              │
                         Preview / Use